## Silver Layer
###### banking.bronze.cbs_branch -> banking.silver.dim_branch (MERGE)

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
bronze_source= "banking.bronze.cbs_branch"
silver_target= "banking.silver.dim_branch"
base="abfss://data@stnajdbanking.dfs.core.windows.net"
checkpoint = f"{base}/silver/dim_branch/checkpoints/"
key= "branch_id"

### Transformation

In [0]:
def transform(df):
    return (
        df.dropDuplicates([key])
        .withColumn("branch_name", trim(col("branch_name")))
        .withColumn("city", initcap(trim(col("city"))))
        .withColumn("staff_count", col("staff_count").cast("int"))
        .withColumn("load_timestamp", current_timestamp())
        .drop("source_file", "bronze_ingested_at")
    )

### Loading into Silver

In [0]:
# Create the empty target once, with the right schema. mode("ignore") does nothing if it already exists,
# so a later run can never overwrite the dimension.
transform(spark.read.table(bronze_source).limit(0)).write.format("delta").mode("ignore").saveAsTable(silver_target)

In [0]:
def upsert(batch_df, batch_id):
    s = batch_df.sparkSession            # Takes one micro-batch of rows from the bronze stream
    batch_df = transform(batch_df)
    if not s.catalog.tableExists(silver_target):
        batch_df.write.format("delta").mode("overwrite").saveAsTable(silver_target)   # first run
    else:
        (DeltaTable.forName(s, silver_target).alias("t")
            .merge(batch_df.alias("s"), f"t.{key} = s.{key}") # Performs a Delta Lake merge on key (customer_id):
            .whenMatchedUpdateAll() # Matching keys are updated
            .whenNotMatchedInsertAll() # New keys are inserted
            .execute())


In [0]:
(
    spark.readStream.table(bronze_source)
    .writeStream
    .foreachBatch(upsert)
    .outputMode("update")
    .option("checkpointLocation", checkpoint) # Stores progress so the stream can resume safely and avoid reprocessing completed batches.
    .trigger(availableNow=True) # Process all currently available data, then stop
    .start()
    .awaitTermination()
)

In [0]:
display(spark.table(silver_target))

In [0]:
#spark.sql("DROP TABLE IF EXISTS banking.silver.dim_branch")

In [0]:
#dbutils.fs.rm("abfss://data@stnajdbanking.dfs.core.windows.net/silver/dim_branch/checkpoints", True)